# EVE vs AdamW -- Race to 80% Macro F1

**Task:** ResNet-18 on CIFAR-10.
Both optimizers use default hyperparameters.
Training stops when validation macro F1 reaches the target or the epoch cap is hit.

In [ ]:
import os, sys, subprocess

BRANCH = "full-batch-probing"

if os.path.exists("/content"):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH,
         "https://github.com/SattamAltwaim/EVE.git", "/content/EVE"],
        capture_output=True, check=False,
    )
    if "/content/EVE" not in sys.path:
        sys.path.insert(0, "/content/EVE")
else:
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "eve_optimizer")) and parent not in sys.path:
        sys.path.insert(0, parent)

from eve_optimizer import EVE
print("EVE imported successfully")

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

## Configuration

In [ ]:
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 128
VAL_BATCH   = 512
NUM_WORKERS = 0
LR          = 1e-3
MAX_EPOCHS  = 80
TARGET_F1   = 0.80
SEED        = 42

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Config : batch={BATCH_SIZE}  val_batch={VAL_BATCH}  lr={LR}  "
      f"target_f1={TARGET_F1}  max_epochs={MAX_EPOCHS}")

## Data

In [ ]:
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_ds = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=True, download=True, transform=train_tf)
val_ds = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=False, download=True, transform=val_tf)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=True)
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=VAL_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

print(f"Train batches: {len(train_dl)}  |  Val samples: {len(val_ds)}")

## Model and evaluation

In [ ]:
def make_model():
    m = torchvision.models.resnet18(weights=None, num_classes=10)
    return m.to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        total_loss += criterion(out, yb).item() * len(yb)
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(yb.cpu().tolist())
    n = len(targets)
    acc = sum(p == t for p, t in zip(preds, targets)) / n
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    return total_loss / n, acc, macro_f1

## Training loop

In [ ]:
def train_race(label, make_opt_fn, *, is_eve=False):
    """Train until val macro F1 >= TARGET_F1 or MAX_EPOCHS."""
    torch.manual_seed(SEED)
    model = make_model()
    loss_fn = nn.CrossEntropyLoss()
    opt = make_opt_fn(model.parameters())

    hist = dict(train_loss=[], val_loss=[], val_acc=[], val_f1=[],
                epoch_wall_s=[])
    race_epoch, race_time = None, None
    wall_start = time.perf_counter()

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss, n_batches = 0.0, 0

        pbar = tqdm(train_dl, desc=f"Ep {epoch:3d}", leave=False,
                    dynamic_ncols=True)
        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            if is_eve:
                opt.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                         current_loss=loss.item())
            else:
                opt.step()
            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        tr_loss = epoch_loss / n_batches
        v_loss, v_acc, v_f1 = evaluate(model, val_dl)
        wall_elapsed = time.perf_counter() - wall_start

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)
        hist["val_f1"].append(v_f1)
        hist["epoch_wall_s"].append(wall_elapsed)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Ep {epoch:3d}/{MAX_EPOCHS}  "
                  f"train_loss={tr_loss:.4f}  "
                  f"val_loss={v_loss:.4f}  "
                  f"val_acc={v_acc:.4f}  "
                  f"val_f1={v_f1:.4f}  "
                  f"[{wall_elapsed/60:.1f} min]")

        if race_epoch is None and v_f1 >= TARGET_F1:
            race_epoch = epoch
            race_time = wall_elapsed
            print(f"\n  *** {label} hit {TARGET_F1:.0%} macro F1 at epoch {epoch} "
                  f"({race_time/60:.2f} min) ***\n")
            break

    if race_epoch is None:
        best_f1 = max(hist["val_f1"])
        print(f"\n  {label} did not reach {TARGET_F1:.0%} F1 in {MAX_EPOCHS} epochs. "
              f"Best val F1: {best_f1:.4f}")

    hist["race_epoch"] = race_epoch
    hist["race_time_s"] = race_time
    return hist

## Run AdamW

In [ ]:
results = {}
results["AdamW"] = train_race(
    "AdamW",
    lambda p: torch.optim.AdamW(p, lr=LR),
    is_eve=False,
)

## Run EVE

In [ ]:
results["EVE"] = train_race(
    "EVE (K=4)",
    lambda p: EVE(p, lr=LR, K=4, record_diagnostics=False),
    is_eve=True,
)

## Race summary

In [ ]:
print("\n" + "=" * 60)
print(f"  RACE SUMMARY -- first to val macro F1 >= {TARGET_F1:.0%}")
print("=" * 60)
for name, h in results.items():
    if h["race_epoch"] is not None:
        print(f"  {name:8s}  epoch {h['race_epoch']:3d}  "
              f"{h['race_time_s']/60:6.2f} min  "
              f"val_f1={max(h['val_f1']):.4f}")
    else:
        print(f"  {name:8s}  did NOT reach target  "
              f"best_f1={max(h['val_f1']):.4f}")

## Plots

In [ ]:
COLORS = {"AdamW": "#2196F3", "EVE": "#E91E63"}

def epochs_axis(h):
    return list(range(1, len(h["train_loss"]) + 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
for name, h in results.items():
    ax.plot(epochs_axis(h), h["train_loss"], color=COLORS[name],
            label=name, linewidth=1.5)
ax.set_title("Train Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, h in results.items():
    ax.plot(epochs_axis(h), h["val_loss"], color=COLORS[name],
            label=name, linewidth=1.5)
ax.set_title("Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
for name, h in results.items():
    ax.plot(epochs_axis(h), [f * 100 for f in h["val_f1"]],
            color=COLORS[name], label=name, linewidth=1.5)
    if h["race_epoch"] is not None:
        ax.axvline(h["race_epoch"], color=COLORS[name],
                   linewidth=0.8, linestyle=":")
ax.axhline(TARGET_F1 * 100, color="grey", linewidth=1.2,
           linestyle="--", label="Target")
ax.set_title("Val Macro F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1 (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("EVE vs AdamW -- ResNet-18 / CIFAR-10",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, h in results.items():
    minutes = [t / 60 for t in h["epoch_wall_s"]]
    ax.plot(minutes, [f * 100 for f in h["val_f1"]],
            color=COLORS[name], label=name, linewidth=1.5)
    if h["race_time_s"] is not None:
        ax.axvline(h["race_time_s"] / 60, color=COLORS[name],
                   linewidth=0.8, linestyle=":")
ax.axhline(TARGET_F1 * 100, color="grey", linewidth=1.2,
           linestyle="--", label="Target")
ax.set_title("Val Macro F1 vs Wall-Clock Time")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Macro F1 (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()